# **Question 5: Intermediate - Resource Management & Context Managers**

In MLOps, we deal with "expensive" resources:
* Opening a connection to a **Feature Store** (Redis/Postgres).
* Loading a 5GB model into GPU memory.
* Opening a file to write logs.

If your code crashes halfway through, and you haven't closed these resources, you get **Memory Leaks** or **Connection Timeouts** (the "Too many open files" error).

**The Scenario:**
You are writing a Python class `FeatureStoreClient` that connects to a database. You want to ensure that *no matter what happens* (even if an error occurs while fetching features), the connection is closed properly.

**The Question:**
1.  Which Python keyword handles this automatically?
2.  If you had to write a custom class to support this keyword, which **two magic methods** (dunder methods) must you implement?
3.  What specific logic goes into the second magic method to handle exceptions that might have occurred inside the block?

---

## Background (Why This Matters in MLOps)

In MLOps and backend systems, we work with **expensive resources**:
- Database / Feature Store connections (Redis, Postgres)
- Large ML models loaded into GPU memory
- File handles (logs, artifacts)

If code crashes midway and these resources are not released properly, it causes:
- Memory leaks
- Connection pool exhaustion
- OS-level errors: `OSError: [Errno 24] Too many open files`

Python provides a built-in mechanism to handle this safely and automatically.

---

## Q1 — Which Python keyword handles this automatically?

**Answer: `with`**

The `with` keyword implements the **context manager protocol**. It wraps the block in an implicit `try/finally`, so cleanup always runs — even if an exception occurs inside the block.

```python
with open("logs.txt", "a") as f:
    f.write("model prediction logged")
# file is always closed here — crash or not
```

### Interview Phrasing
> "The `with` statement implements the context manager protocol. It wraps the block in an implicit try/finally, so the cleanup code always runs — even if an unhandled exception occurs inside the block."

---

## Q2 — Which Two Dunder Methods Must You Implement?

| Method | Role |
|---|---|
| `__enter__` | Acquires the resource. Its return value is bound to the `as` variable. |
| `__exit__` | Always called on the way out. Releases the resource. Controls exception propagation. |

```python
class FeatureStoreClient:
    def __enter__(self):
        self.conn = connect_to_db()
        return self            # bound to 'as client'

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.conn.close()     # ALWAYS runs
        return False          # re-raise any exception

with FeatureStoreClient() as client:
    features = client.get_features(user_id)
```

---

## Q3 — What Logic Goes Inside `__exit__`? (Most Important)

### Method Signature
```python
def __exit__(self, exc_type, exc_val, exc_tb):
```

### The Three Parameters

| Parameter | Meaning | Value if no exception |
|---|---|---|
| `exc_type` | The exception *class* (e.g. `ValueError`) | `None` |
| `exc_val` | The exception *instance* with the message | `None` |
| `exc_tb` | The traceback object (stack frames) | `None` |

### Return Value Controls Exception Propagation

| Return Value | Behaviour | When to use |
|---|---|---|
| `False` / `None` | Exception **re-raises** (propagates) | 99% of cases — you cleaned up but don't hide the error |
| `True` | Exception **suppressed** | Rare — deliberate dry-run or retry wrappers |

### Full Example
```python
def __exit__(self, exc_type, exc_val, exc_tb):
    # Step 1: always clean up first
    self.conn.close()

    # Step 2: optionally inspect the exception
    if exc_type is ConnectionError:
        log_alert(f"DB dropped: {exc_val}")

    # Step 3: return False → exception propagates (default)
    return False
```

---

## Internal Execution Flow (What Python Actually Does)

```python
# What you write:
with FeatureStoreClient() as client:
    client.get_features(user_id)

# What Python executes under the hood:
_mgr = FeatureStoreClient()
client = _mgr.__enter__()
try:
    client.get_features(user_id)
except:
    if not _mgr.__exit__(*sys.exc_info()):
        raise           # re-raise if __exit__ returns False
else:
    _mgr.__exit__(None, None, None)
```

> The `try/except/else` is why cleanup is truly guaranteed — and why the three parameters exist.

---

## Senior Engineer Answer — `contextlib.contextmanager`

For simple cases, writing a full class is overkill. `contextlib.contextmanager` turns a **generator function** into a context manager.

- Everything **before** `yield` → `__enter__` logic
- Everything **after** `yield` (in `finally`) → `__exit__` logic

```python
from contextlib import contextmanager

@contextmanager
def feature_store_connection():
    conn = connect_to_db()       # __enter__ logic
    try:
        yield conn               # body of 'with' block runs here
    finally:
        conn.close()             # __exit__ logic — always runs

with feature_store_connection() as conn:
    features = conn.get_features(user_id)
```

### MLOps Use Case — Temporarily Change Working Directory for Model Saving
```python
@contextmanager
def model_save_dir(path):
    original = os.getcwd()
    os.makedirs(path, exist_ok=True)
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(original)    # always restore cwd

with model_save_dir("/models/v2"):
    model.save("weights.pt")
```

---

## One-Line Revision Summary

> `with` triggers the context manager protocol → `__enter__` acquires the resource → the block runs inside an implicit `try` → `__exit__(exc_type, exc_val, exc_tb)` *always* runs cleanup → returning `False` re-raises, returning `True` suppresses the exception.

---

## Interview Delivery Tips

1. **Lead with the *why*:** "In MLOps we have expensive resources like GPU memory and connection pools; if code crashes, we get memory leaks and connection timeouts. Python's context manager protocol solves this."
2. **The critical gap most candidates miss:** The `__exit__` return value — `False` propagates, `True` suppresses. Always mention this.
3. **Senior signal:** Mention `contextlib.contextmanager` — it shows real production experience.
4. **Connect to real consequences:** Leaked DB connections cause services to slow down, databases reject connections, and pods crash in Kubernetes.

## ✅ Example Code

```python
class FeatureStoreClient:
    def connect(self):
        print("Connected")

    def get_features(self):
        print("Getting features")
        raise Exception("Error happened!")

    def close(self):
        print("Connection closed")

    def __enter__(self):
        print("Entering context")
        self.connect()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Inside exit")
        if exc_type:
            print(f"Error handled: {exc_value}")
        self.close()
        return False  # re-raise exception


# Usage
with FeatureStoreClient() as client:
    client.get_features()
```

---

# 🧾 Output

```
Entering context
Connected
Getting features
Inside exit
Error handled: Error happened!
Connection closed
Exception: Error happened!
```

---

# 🔥 Step-by-Step Execution Flow

---

## 🟢 Step 1: `with` starts

Python creates object:

```python
obj = FeatureStoreClient()
```

---

## 🟢 Step 2: `__enter__()` is called

```python
obj.__enter__()
```

### Logs:

```
Entering context
Connected
```

👉 Resource is acquired (connection opened)

---

## 🟡 Step 3: Inside `with` block

```python
client.get_features()
```

### Logs:

```
Getting features
```

💥 Exception occurs here:

```
Exception("Error happened!")
```

👉 Execution stops and jumps to `__exit__`

---

## 🔴 Step 4: `__exit__()` is called (VERY IMPORTANT)

```python
obj.__exit__(exc_type, exc_value, traceback)
```

### Logs:

```
Inside exit
Error handled: Error happened!
Connection closed
```

👉 Cleanup happens here **ALWAYS**

* Connection closed
* Error received as parameters

---

## 🔴 Step 5: Exception is re-raised

Because:

```python
return False
```

👉 Python re-throws the exception

### Final Log:

```
Exception: Error happened!
```

---

# 🎯 Final Flow Summary (One Look Revision)

```
1. __enter__()       → Entering context
                     → Connected

2. with block        → Getting features
                     → ❌ Exception occurs

3. __exit__()        → Inside exit
                     → Error handled: Error happened!
                     → Connection closed

4. Exception raised  → Exception: Error happened!
```

---